In [1]:
## Cell 1: Cài đặt thư viện
!pip install -q --force-reinstall "pillow<12"
import os
os.chdir("/kaggle/working")

if not os.path.isdir("/kaggle/working/CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git

%cd /kaggle/working/CatVTON
!grep -vi '^gradio' requirements.txt > requirements_notebook.txt
!pip install -q -r requirements_notebook.txt
!pip install -q huggingface_hub
!pip install -q fvcore iopath yacs pycocotools omegaconf cloudpickle av

# Sửa lỗi GPU
!pip install -q --force-reinstall torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121

# Cài thêm các tool làm API
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio

print(">>> Cài xong! Nhớ Restart Kernel nhé!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
/kaggle/working/CatVTON
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Cannot install -r requirements_notebook.txt (line 14), -r requirements_notebook.txt (line 3), -r requirements_notebook.txt (line 4) and huggingface_hub<2.0 and >=0.34.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 100.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━

In [2]:
## Cell 2: Tải Mô hình vào RAM (Chỉ chạy 1 lần)
import os, time, io
import torch
from PIL import Image
from diffusers.image_processor import VaeImageProcessor
import transformers.utils.import_utils as _tf_import_utils
_tf_import_utils.check_torch_load_is_safe = lambda: None
from huggingface_hub import snapshot_download

from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

# Tải model CatVTON
repo_path = snapshot_download(repo_id="zhengchong/CatVTON")

pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device="cuda",
    skip_safety_check=True
)

mask_processor = VaeImageProcessor(
    vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device="cuda",
)

print(">>> Trái tim AI đã nổ máy! Sẵn sàng nhận lệnh.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/748 [00:00<?, ?B/s]

An error occurred while trying to fetch booksforcharlie/stable-diffusion-inpainting: booksforcharlie/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


unet/diffusion_pytorch_model.bin:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

>>> Trái tim AI đã nổ máy! Sẵn sàng nhận lệnh.


Trước khi chạy, bạn cần lên trang [ngrok.com](https://ngrok.com/), đăng ký 1 tài khoản miễn phí, vào mục **Your Authtoken** để lấy dải mã token bí mật của bạn và dán vào dòng `NGROK_TOKEN` bên dưới nhé.

In [ ]:
## Cell 3: Bật Server API & Ngrok (Chạy liên tục)
!pip install -q -U nest_asyncio
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

# Bật CORS để cho phép Web ở Localhost gọi lên không bị chặn
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

@app.post("/api/tryon")
async def tryon_api(person_image: UploadFile = File(...), cloth_image: UploadFile = File(...)):
    print(f"\n[+] Đang nhận đơn hàng: Người ({person_image.filename}) - Áo ({cloth_image.filename})")
    
    # 1. Đọc ảnh từ mạng gửi sang
    person_bytes = await person_image.read()
    cloth_bytes = await cloth_image.read()
    person_img = Image.open(io.BytesIO(person_bytes)).convert("RGB")
    cloth_img = Image.open(io.BytesIO(cloth_bytes)).convert("RGB")
    
    # 2. Xử lý ảnh
    WIDTH, HEIGHT = 768, 1024
    person_img = resize_and_crop(person_img, (WIDTH, HEIGHT))
    cloth_img = resize_and_padding(cloth_img, (WIDTH, HEIGHT))
    
    # 3. Chế tạo mặt nạ
    print("    -> Đang lấy số đo...")
    mask = automasker(person_img, "upper")["mask"]
    mask = mask_processor.blur(mask, blur_factor=9)
    
    # 4. Gọi AI ghép đồ
    print("    -> Đang may áo... (Chờ khoảng 40s)")
    result = pipeline(
        image=person_img,
        condition_image=cloth_img,
        mask=mask,
        num_inference_steps=25, # Đã giảm xuống 25 steps để chạy nhanh gấp đôi
        guidance_scale=2.5,
    )[0]
    
    # 5. Lưu và trả hàng
    out_path = os.path.join(OUTPUT_DIR, f"result_{int(time.time())}.png")
    result.save(out_path)
    print("    -> Hoàn tất! Đang gửi ảnh về cho Web.")
    
    return FileResponse(out_path, media_type="image/png")


# ======== KẾT NỐI NGROK ========
NGROK_TOKEN = "3B91485xasWPb3CYuMwP0qb76S7_7vgxKozCzYTYdnDCtaCFY"
ngrok.set_auth_token(NGROK_TOKEN)

# Đóng các tunnel cũ nếu có
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(8000).public_url
print("="*60)
print(f"🚀 API CỦA BẠN ĐÃ SẴN SÀNG TẠI: {public_url}/api/tryon")
print(f"👉 Copy nguyên cái link {public_url} này dán vào Web Local nhé!")
print("="*60)

# Chạy server
import asyncio
nest_asyncio.apply()

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

🚀 API CỦA BẠN ĐÃ SẴN SÀNG TẠI: https://cactaceous-tatum-semiadhesively.ngrok-free.dev/api/tryon
👉 Copy nguyên cái link https://cactaceous-tatum-semiadhesively.ngrok-free.dev này dán vào Web Local nhé!


INFO:     Started server process [660]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     118.69.7.9:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     118.69.7.9:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     118.69.7.9:0 - "GET /api/tryon HTTP/1.1" 405 Method Not Allowed
INFO:     118.69.7.9:0 - "GET / HTTP/1.1" 404 Not Found
